# CASMI26 hybrid: library retrieval + CheMM-R1 + ChemMRL (v3, time-budgeted)

**Runtime design (target < 4 h, hard deadline 3.75 h):**
1. Library build streams `train.parquet` (several spectra per structure, vectorised modified-cosine matching).
2. Retrieval for all molecules, then a **preliminary submission is written immediately**.
3. CheMM-R1 runs on both GPUs with batched generation; the token cap is recomputed before every batch from the time left, so generation always stops with `POST_RESERVE_MIN` to spare. Results are checkpointed to `chemm_cache.jsonl`.
4. ChemMRL embeds only what it needs, then final fusion overwrites `submission.csv`.

Tunable knobs live in the config cell: `HARD_DEADLINE_HOURS`, `POST_RESERVE_MIN`, `CHEMM_BATCH_SIZE`, `CHEMM_MIN_TOKENS/MAX_TOKENS`, `CHEMM_TEMPERATURE`.

In [ ]:
!pip install --no-index --find-links /kaggle/input/datasets/eveliaveldrine/dependencies \
    "transformers>=4.56.1,<5.0.0" "accelerate" "qwen-vl-utils" "pillow"
!pip install --no-index --find-links /kaggle/input/datasets/eveliaveldrine/dependencies \
    "chem-mrl>=0.7.3" --no-deps
!pip install -q --no-index --find-links /kaggle/input/datasets/eveliaveldrine/rdkit-wheels rdkit==2025.3.6

In [ ]:
import os, gc, io, re, json, time, math, random, warnings, threading, queue
from collections import defaultdict
from functools import lru_cache

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

from rdkit import RDLogger, DataStructs
from rdkit.Chem import (MolFromSmiles, MolToSmiles, MolToInchiKey,
                        rdMolDescriptors, rdFingerprintGenerator)
RDLogger.DisableLog("rdApp.*")
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ======================= GLOBAL TIME BUDGET =======================
# Everything is scheduled against one wall-clock budget. CheMM-R1 generation
# (the only open-ended step) stops early enough that ChemMRL + fusion + writing
# the submission always fit before the hard deadline.
NOTEBOOK_START = time.time()
HARD_DEADLINE_HOURS = 3.75                      # 15 min safety margin below 4 h
HARD_DEADLINE_SECONDS = HARD_DEADLINE_HOURS * 3600
POST_RESERVE_MIN = 20                           # kept for ChemMRL + fusion + write
CHEMM_STOP_AT = HARD_DEADLINE_SECONDS - POST_RESERVE_MIN * 60

def elapsed_min():
    return (time.time() - NOTEBOOK_START) / 60

def remaining_s():
    return HARD_DEADLINE_SECONDS - (time.time() - NOTEBOOK_START)

def deadline_check(phase, reserve_seconds=900):
    print(f"[{phase}] elapsed={elapsed_min():.1f}min remaining={remaining_s()/60:.1f}min")
    return remaining_s() > reserve_seconds

# ======================= PATHS =======================
COMP_DIRS = [
    "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra",
    "/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra",
    "./data", ".",
]
COMP_DIR = next((p for p in COMP_DIRS if os.path.exists(p)), COMP_DIRS[0])
TEST_PATH  = os.path.join(COMP_DIR, "test.parquet")
TRAIN_PATH = os.path.join(COMP_DIR, "train.parquet")
SAMPLE_SUB = os.path.join(COMP_DIR, "sample_submission.csv")
WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "."
SUBMISSION_PATH = os.path.join(WORKING_DIR, "submission.csv")
DEBUG_PATH = os.path.join(WORKING_DIR, "chemm_hybrid_debug.json")
CHEMM_CACHE_PATH = os.path.join(WORKING_DIR, "chemm_cache.jsonl")   # resume file

MODEL_ID = "/kaggle/input/datasets/eveliaveldrine/zzha6204-chemm-r1-offline"

# ======================= CheMM-R1 =======================
USE_CHEMM = True
CHEMM_MAX_GPUS = 2
CHEMM_BATCH_SIZE = 6              # molecules per generate() call, per GPU
CHEMM_MIN_TOKENS = 640            # token cap is chosen adaptively in [MIN, MAX]
CHEMM_MAX_TOKENS = 2048           #   from the time that is left (see generation cell)
CHEMM_RESCUE_TOKENS = 96          # if the cap cuts the reasoning, force "<answer>" and finish
CHEMM_TEMPERATURE = 0.6           # <= 0 means greedy
CHEMM_TOP_P = 0.95
CHEMM_TOP_K = 20
CHEMM_SKIP_IF_SPECTRAL_ABOVE = 0.95   # library match this good needs no LLM call
CHEMM_MOLECULES_CAP = None            # None = as many as the time budget allows

# ======================= ChemMRL =======================
USE_CHEMMRL = True
CHEMMRL_MODEL_ID = "Derify/ChemMRL"
CHEMMRL_BATCH_SIZE = 64
CHEMMRL_MAX_ANCHORS = 2
CHEMMRL_EMB_DIM = 256
CHEMMRL_LOCAL_PATHS = [
    "/kaggle/input/datasets/eveliaveldrine/derify-chemmrl-offline",
    "/kaggle/input/chem-mrl",
    "./chemmrl",
]

# ======================= Retrieval =======================
MAX_STRUCTURES_FOR_LIBRARY = 600_000
MAX_SPECTRA_PER_STRUCT = 3        # keep several train spectra per structure (different adducts/energies)
MASS_CANDIDATE_LIMIT = 150
MAX_QUERY_SPECTRA_SCORING = 8
PRIMARY_PPM = 10.0
FALLBACK_PPM = 30.0
FINAL_K = 25
MAX_QUERY_PEAKS = 128
PEAK_TOL = 0.02
STRONG_SPECTRAL = 0.70            # spectral match this strong is guaranteed a top slot

# ======================= Fusion weights (unchanged) =======================
W_MASS          = 0.15
W_SPECTRAL      = 0.35
W_CHEMMRL       = 0.20
W_PRIOR         = 0.05
BONUS_CHEMM_R1  = 0.20
SOURCE_BONUS = {"chemm_r1": 0.35, "train_retrieval": 0.0,
                "mass_fallback": -0.05, "nearest_mass": -0.05, "both": 0.40}

print("COMP_DIR:", COMP_DIR)
print("MODEL_ID:", MODEL_ID)
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name} | VRAM {p.total_memory/1e9:.1f} GB | sm_{p.major}{p.minor}")

In [ ]:
import sys, importlib, subprocess

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")

tv = importlib.util.find_spec("torchvision")
if tv is None:
    print("[WARN] torchvision missing. Transformers AutoProcessor may fail.")
else:
    try:
        import torchvision
        print(f"torchvision: {torchvision.__version__}")
    except Exception as e:
        print(f"[WARN] torchvision broken: {e!r}")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"],
                       check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print("[INFO] Removed broken torchvision. Restart the kernel before continuing.")

try:
    import transformers
    print(f"Transformers: {transformers.__version__}")
    mm = tuple(int(x) for x in transformers.__version__.split(".")[:2])
    if not ((4, 56) <= mm < (5, 0)):
        # warn instead of aborting the whole run
        print(f"[WARN] Transformers {transformers.__version__} is outside the ChemMRL-supported 4.56-4.x range")
except Exception as e:
    print(f"[WARN] transformers import failed: {e!r}")
    USE_CHEMM = False

try:
    import sentence_transformers
    print(f"SentenceTransformers: {sentence_transformers.__version__}")
except Exception as e:
    print(f"[WARN] SentenceTransformers import failed: {e!r}")
    USE_CHEMMRL = False

In [ ]:
ADDUCT_DELTAS = {
    "[M+H]+":          1.007276466621,
    "[M+NH4]+":        18.033823005,
    "[M-H2O+H]+":      -17.003288036,
    "[M-2H2O+H]+":     -35.013853101,
    "[M+Na]+":         22.989218,
    "[M+K]+":          38.963158,
    "[M-H]-":          -1.007276466621,
    "[M-H2O-H]-":      -19.0178415,
    "[M+CH2O2-H]-":    44.998201,
    "[M+Cl]-":         34.969402,
    "[M+FA-H]-":       44.998201,
    "[M+HCOO]-":       44.998201,
    "[M+Ac-H]-":       59.013851,
    "[M+CH3COO]-":     59.013851,
}

def neutral_mass_from_adduct(precursor_mz, adduct):
    adduct = str(adduct).strip()
    if adduct not in ADDUCT_DELTAS:
        raise ValueError(f"Unsupported CASMI adduct: {adduct}")
    return float(precursor_mz) - ADDUCT_DELTAS[adduct]

def safe_neutral_mass(precursor_mz, adduct):
    try:
        return neutral_mass_from_adduct(precursor_mz, adduct)
    except Exception:
        return np.nan

def ppm_error(mass, target):
    return abs(float(mass) - float(target)) / max(abs(float(target)), 1e-9) * 1e6

# Plain InChIKey connectivity block, cached. (The old tautomer-canonicalising key was
# the slowest step in the notebook -- it ran for every candidate of every molecule --
# and it did not match the InChIKey14 the scorer computes from the submitted SMILES.)
@lru_cache(maxsize=1_000_000)
def ikey14(smi):
    mol = MolFromSmiles(smi) if isinstance(smi, str) else None
    if mol is None:
        return None
    try:
        k = MolToInchiKey(mol)
        return k.split("-")[0] if k else None
    except Exception:
        return None

def structure_prior_from_mol(mol):
    heavy = mol.GetNumHeavyAtoms()
    rings = rdMolDescriptors.CalcNumRings(mol)
    hetero = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() in (7, 8, 15, 16, 17))
    return float(0.002 * min(heavy, 80) + 0.01 * min(rings, 8) + 0.003 * min(hetero, 12))

print("Supported adducts:", len(ADDUCT_DELTAS))

In [ ]:
test_df = pd.read_parquet(TEST_PATH)
required = {"molecule_id", "precursor_mz", "adduct",
            "ms2_mzs", "ms2_normalized_intensities"}
missing = required - set(test_df.columns)
if missing:
    raise RuntimeError(f"Missing required columns: {missing}")

test_df["neutral_mass"] = [
    safe_neutral_mass(mz, ad)
    for mz, ad in zip(test_df["precursor_mz"], test_df["adduct"])
]

print(f"Test spectra: {len(test_df):,}")
print(f"Test molecules: {test_df.molecule_id.nunique():,}")
print(f"Unsupported adduct rows: {test_df['neutral_mass'].isna().sum()}")

molecule_meta = []
for mid, g in test_df.groupby("molecule_id", sort=False):
    masses = g["neutral_mass"].dropna().to_numpy(dtype=np.float64)
    if len(masses) == 0:
        continue
    med = float(np.median(masses))
    molecule_meta.append({
        "molecule_id": mid,
        "target_mass": med,
        "mass_spread_ppm": float((masses.max() - masses.min()) / max(med, 1e-9) * 1e6)
                           if len(masses) > 1 else 0.0,
        "n_spectra": len(g),
    })
molecule_meta = pd.DataFrame(molecule_meta)
target_mass_by_mid = dict(zip(molecule_meta["molecule_id"], molecule_meta["target_mass"]))
print(f"Molecules prepared: {len(molecule_meta)}")
print(molecule_meta["mass_spread_ppm"].describe()[["mean", "50%", "max"]])

In [ ]:
_EMPTY = (np.empty(0, np.float32), np.empty(0, np.float32))

def select_peaks(mzs, intensities, max_peaks=MAX_QUERY_PEAKS, noise_floor=0.005):
    """Clean a spectrum: finite/positive peaks, base-peak normalised, noise filtered,
    top-N by intensity, and ALWAYS sorted by m/z (the matcher needs sorted m/z; the old
    code only preserved input order)."""
    if mzs is None or intensities is None:
        return _EMPTY
    mzs = np.asarray(mzs, dtype=np.float64)
    ints = np.asarray(intensities, dtype=np.float64)
    n = min(mzs.size, ints.size)
    if n == 0:
        return _EMPTY
    mzs, ints = mzs[:n], ints[:n]
    ok = np.isfinite(mzs) & np.isfinite(ints) & (ints > 0)
    mzs, ints = mzs[ok], ints[ok]
    if ints.size == 0:
        return _EMPTY
    ints = ints / ints.max()
    keep = ints >= noise_floor
    mzs, ints = mzs[keep], ints[keep]
    if mzs.size > max_peaks:
        top = np.argsort(-ints, kind="stable")[:max_peaks]
        mzs, ints = mzs[top], ints[top]
    order = np.argsort(mzs, kind="stable")
    return mzs[order].astype(np.float32), ints[order].astype(np.float32)

def to_weights(ints):
    """sqrt-intensity, L2-normalised (down-weights dominant peaks, standard for MS/MS cosine)."""
    w = np.sqrt(ints.astype(np.float64))
    n = np.linalg.norm(w)
    return (w / n).astype(np.float32) if n > 0 else w.astype(np.float32)

def _nearest(l_mz, x, tol):
    """For every x find the nearest peak in sorted l_mz; return (index, within_tol)."""
    n = l_mz.shape[0]
    pos = np.searchsorted(l_mz, x)
    lo = np.clip(pos - 1, 0, n - 1)
    hi = np.clip(pos, 0, n - 1)
    dlo = np.abs(l_mz[lo] - x)
    dhi = np.abs(l_mz[hi] - x)
    use_hi = dhi < dlo
    return np.where(use_hi, hi, lo), np.minimum(dlo, dhi) <= tol

def spectral_similarity(q_mz, q_w, l_mz, l_w, shift=0.0, tol=PEAK_TOL):
    """(Modified) cosine between two sorted spectra, vectorised, one-to-one matching.
    shift = precursor difference; peaks may match directly or shifted (modified cosine),
    which keeps different adducts / close analogs comparable."""
    if q_mz.size == 0 or l_mz.size == 0:
        return 0.0
    qi_parts, lj_parts, pr_parts = [], [], []
    j, ok = _nearest(l_mz, q_mz, tol)
    if ok.any():
        qi = np.flatnonzero(ok)
        qi_parts.append(qi); lj_parts.append(j[ok]); pr_parts.append(q_w[qi] * l_w[j[ok]])
    if abs(shift) > tol:
        j2, ok2 = _nearest(l_mz, q_mz - shift, tol)
        if ok2.any():
            qi = np.flatnonzero(ok2)
            qi_parts.append(qi); lj_parts.append(j2[ok2]); pr_parts.append(q_w[qi] * l_w[j2[ok2]])
    if not qi_parts:
        return 0.0
    qi = np.concatenate(qi_parts); lj = np.concatenate(lj_parts); pr = np.concatenate(pr_parts)
    if len(qi_parts) > 1 or np.unique(lj).size < lj.size:
        # greedy one-to-one assignment: best products first, each peak used once
        o = np.argsort(-pr, kind="stable"); qi, lj, pr = qi[o], lj[o], pr[o]
        _, f = np.unique(qi, return_index=True); lj, pr = lj[f], pr[f]
        o = np.argsort(-pr, kind="stable"); lj, pr = lj[o], pr[o]
        _, f = np.unique(lj, return_index=True); pr = pr[f]
    return float(min(pr.sum(), 1.0))

# ---------------- spectrum image for CheMM-R1 (identical look to the original) ----------------
def render_ms2_png(args):
    mzs, intensities, precursor_mz, adduct = args
    fig, ax = plt.subplots(figsize=(7, 3.5), dpi=90)
    if len(mzs):
        ax.vlines(mzs, 0, intensities, linewidth=0.8)
    ax.axvline(precursor_mz, linestyle="--", linewidth=1.0)
    ax.set_xlabel("m/z"); ax.set_ylabel("relative intensity")
    ax.set_ylim(0, 1.08)
    ax.set_xlim(0, max(precursor_mz * 1.05, 100))
    ax.set_title(f"MS2 | {adduct} | precursor={precursor_mz:.4f}")
    ax.tick_params(labelsize=7)
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    plt.close(fig)
    return buf.getvalue()

def png_to_image(b):
    return Image.open(io.BytesIO(b)).convert("RGB")

In [ ]:
def prepare_query_spectra(group):
    prepared = []
    for mz_arr, int_arr, pmz, ad, nm in zip(group["ms2_mzs"], group["ms2_normalized_intensities"],
                                           group["precursor_mz"], group["adduct"], group["neutral_mass"]):
        if not np.isfinite(nm):
            continue
        mz, it = select_peaks(mz_arr, int_arr)
        if len(mz) < 3:
            continue
        prepared.append({"mz": mz, "int": it, "w": to_weights(it),
                         "precursor_mz": float(pmz), "prec": float(pmz),
                         "adduct": str(ad), "neutral_mass": float(nm)})
    prepared.sort(key=lambda x: len(x["mz"]), reverse=True)

    # up to 4 spectra for the CheMM-R1 image prompt, preferring distinct adducts
    selected_idx, seen_adducts = set(), set()
    for i, item in enumerate(prepared):
        if item["adduct"] not in seen_adducts:
            selected_idx.add(i); seen_adducts.add(item["adduct"])
        if len(selected_idx) >= 4:
            break
    if len(selected_idx) < min(4, len(prepared)):
        for i in range(len(prepared)):
            selected_idx.add(i)
            if len(selected_idx) >= 4:
                break
    return prepared, [prepared[i] for i in sorted(selected_idx)]

query_cache = {}
for mid, g in tqdm(test_df.groupby("molecule_id", sort=False), desc="Preparing queries"):
    all_s, visual_s = prepare_query_spectra(g)
    query_cache[mid] = {"all": all_s, "visual": visual_s}

print("Prepared queries for", len(query_cache), "molecules |",
      "without usable spectra:", sum(1 for v in query_cache.values() if not v["all"]))

In [ ]:
# Stream the training parquet in batches (no giant DataFrame), keep up to
# MAX_SPECTRA_PER_STRUCT spectra per structure, store spectra as compact float32 arrays.
print("Building spectral library from train...")
t0 = time.time()
pf = pq.ParquetFile(TRAIN_PATH)
train_cols = set(pf.schema_arrow.names)
cols = ["inchikey14", "normalized_smiles", "ms2_mzs", "ms2_normalized_intensities"]
HAS_PREC = "precursor_mz" in train_cols
if HAS_PREC:
    cols.append("precursor_mz")      # enables modified-cosine matching across adducts
print(f"Train rows: {pf.metadata.num_rows:,} | precursor_mz available: {HAS_PREC}")

struct_index = {}                    # inchikey14 -> structure idx (-1 = unparsable)
lib_smiles, lib_ikey, lib_mass_l, lib_prior_l, struct_specs = [], [], [], [], []
spec_mz, spec_w, spec_prec = [], [], []

pbar = tqdm(total=pf.metadata.num_rows, desc="Indexing train")
for batch in pf.iter_batches(batch_size=20_000, columns=cols):
    df = batch.to_pandas()
    pbar.update(len(df))
    prec_col = df["precursor_mz"].to_numpy() if HAS_PREC else None
    for r, (ikey, smi, mzs, ints) in enumerate(zip(df["inchikey14"], df["normalized_smiles"],
                                                   df["ms2_mzs"], df["ms2_normalized_intensities"])):
        if not isinstance(smi, str) or not isinstance(ikey, str) or not ikey:
            continue
        idx = struct_index.get(ikey)
        if idx is None:
            if len(lib_smiles) >= MAX_STRUCTURES_FOR_LIBRARY:
                continue
            mol = MolFromSmiles(smi)
            if mol is None:
                struct_index[ikey] = -1
                continue
            try:
                mw = rdMolDescriptors.CalcExactMolWt(mol)
                prior = structure_prior_from_mol(mol)
            except Exception:
                struct_index[ikey] = -1
                continue
            idx = len(lib_smiles)
            struct_index[ikey] = idx
            lib_smiles.append(smi); lib_ikey.append(ikey)
            lib_mass_l.append(mw); lib_prior_l.append(prior); struct_specs.append([])
        elif idx < 0:
            continue
        if len(struct_specs[idx]) >= MAX_SPECTRA_PER_STRUCT:
            continue
        mz, it = select_peaks(mzs, ints)
        if len(mz) < 3:
            continue
        sid = len(spec_mz)
        spec_mz.append(mz); spec_w.append(to_weights(it))
        spec_prec.append(float(prec_col[r]) if HAS_PREC else float("nan"))
        struct_specs[idx].append(sid)
    del df
pbar.close()
del struct_index; gc.collect()

if not lib_smiles:
    raise RuntimeError("Library is empty - check TRAIN_PATH / column names")

# sort structures by mass for binary-search lookups
order = np.argsort(np.asarray(lib_mass_l, dtype=np.float64), kind="stable")
lib_mass  = np.asarray(lib_mass_l, dtype=np.float64)[order]
lib_prior = np.asarray(lib_prior_l, dtype=np.float32)[order]
lib_smiles = [lib_smiles[i] for i in order]
lib_ikey   = [lib_ikey[i] for i in order]
struct_specs = [struct_specs[i] for i in order]
spec_prec = np.asarray(spec_prec, dtype=np.float64)
del lib_mass_l, lib_prior_l, order; gc.collect()

n_with_spec = sum(1 for s in struct_specs if s)
print(f"Library: {len(lib_smiles):,} structures | {len(spec_mz):,} spectra "
      f"({n_with_spec:,} structures with >=1 spectrum) | {(time.time()-t0)/60:.1f} min")
print(f"Elapsed: {elapsed_min():.1f} min")

In [ ]:
_fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

@lru_cache(maxsize=200_000)
def get_fp(smi):
    m = MolFromSmiles(smi)
    return _fp_gen.GetFingerprint(m) if m is not None else None

def fp_similarity(smi_a, smi_b):
    a, b = get_fp(smi_a), get_fp(smi_b)
    if a is None or b is None:
        return 0.0
    return float(DataStructs.TanimotoSimilarity(a, b))

def candidate_spectral_score(idx, qspecs):
    """Best (modified) cosine between any train spectrum of this structure and any query spectrum."""
    best = 0.0
    for sid in struct_specs[idx]:
        lmz, lw, lp = spec_mz[sid], spec_w[sid], spec_prec[sid]
        for q in qspecs:
            shift = (q["prec"] - lp) if np.isfinite(lp) else 0.0
            s = spectral_similarity(q["mz"], q["w"], lmz, lw, shift)
            if s > best:
                best = s
    return best

def _window(target_mass, ppm):
    d = target_mass * ppm * 1e-6
    lo = int(np.searchsorted(lib_mass, target_mass - d, side="left"))
    hi = int(np.searchsorted(lib_mass, target_mass + d, side="right"))
    return lo, hi

def _closest(lo, hi, target_mass, limit):
    idx = np.arange(lo, hi)
    if idx.size == 0:
        return idx
    return idx[np.argsort(np.abs(lib_mass[idx] - target_mass), kind="stable")[:limit]]

def retrieve_candidates(target_mass, qspecs):
    """Mass-window candidates from the library, scored by spectral similarity.
    Always tries to give >= FINAL_K candidates (widens to FALLBACK_PPM, then nearest mass)
    so the 25 guess slots are never wasted."""
    qs = sorted(qspecs, key=lambda s: -len(s["mz"]))[:MAX_QUERY_SPECTRA_SCORING]
    chosen = {}
    lo, hi = _window(target_mass, PRIMARY_PPM)
    for i in _closest(lo, hi, target_mass, MASS_CANDIDATE_LIMIT):
        chosen[int(i)] = "train_retrieval"
    if len(chosen) < FINAL_K:
        lo, hi = _window(target_mass, FALLBACK_PPM)
        for i in _closest(lo, hi, target_mass, MASS_CANDIDATE_LIMIT):
            if len(chosen) >= MASS_CANDIDATE_LIMIT:
                break
            chosen.setdefault(int(i), "mass_fallback")
    if not chosen:
        pos = int(np.searchsorted(lib_mass, target_mass))
        lo, hi = max(0, pos - FINAL_K), min(len(lib_mass), pos + FINAL_K)
        for i in _closest(lo, hi, target_mass, FINAL_K):
            chosen[int(i)] = "nearest_mass"
    out = []
    for i, src in chosen.items():
        mw = float(lib_mass[i])
        out.append({
            "idx": i, "smiles": lib_smiles[i], "mass": mw,
            "ppm": ppm_error(mw, target_mass),
            "spectral_score": candidate_spectral_score(i, qs) if qs else 0.0,
            "prior_score": float(lib_prior[i]), "source": src,
        })
    return out

_mid0 = molecule_meta.iloc[0]["molecule_id"]
_t = time.time()
_ex = retrieve_candidates(float(molecule_meta.iloc[0]["target_mass"]), query_cache[_mid0]["all"])
print(f"Example: {len(_ex)} candidates in {time.time()-_t:.2f}s | "
      f"max spectral_score={max((c['spectral_score'] for c in _ex), default=0):.3f}")

In [ ]:
# ChemMRL state (filled later; fusion works without it)
chem_mrl = None
CHEMMRL_ACTIVE = False
CHEMMRL_API = None
chem_mrl_embeddings = {}

def dense_tanimoto(a, b):
    dot = float(np.dot(a, b))
    denom = float(np.dot(a, a) + np.dot(b, b) - dot)
    return float(np.clip(dot / denom, 0.0, 1.0)) if denom > 1e-12 else 0.0

def anchor_similarity(smi, anchors):
    """Similarity of a candidate to CheMM-R1's own predictions: ChemMRL embedding Tanimoto
    when available, Morgan Tanimoto when the ChemMRL channel is off."""
    if not anchors:
        return 0.0
    if CHEMMRL_ACTIVE and chem_mrl_embeddings:
        v = chem_mrl_embeddings.get(smi)
        if v is None:
            return 0.0
        scores = [dense_tanimoto(v, chem_mrl_embeddings[a]) for a in anchors if a in chem_mrl_embeddings]
        return max(scores) if scores else 0.0
    return max(fp_similarity(smi, a) for a in anchors)

def make_chemm_entry(smi, target_mass):
    mol = MolFromSmiles(smi)
    if mol is None:
        return None
    mw = rdMolDescriptors.CalcExactMolWt(mol)
    return {"idx": None, "smiles": MolToSmiles(mol), "mass": mw, "ppm": ppm_error(mw, target_mass),
            "spectral_score": 0.0, "prior_score": structure_prior_from_mol(mol), "source": "chemm_r1"}

def fuse_candidates(target_mass, pool, chemm_smiles, anchors):
    entries = [dict(e, chemm_exact=False) for e in pool]

    chemm_keys = {}
    for s in chemm_smiles:
        k = ikey14(s)
        if k:
            chemm_keys.setdefault(k, s)
    if chemm_keys:
        have = set()
        for e in entries:
            k = ikey14(e["smiles"]) or e["smiles"]
            e["key"] = k
            have.add(k)
            if k in chemm_keys:                      # CheMM-R1 agrees with a library structure
                e["source"] = "both"; e["chemm_exact"] = True
        for k, s in chemm_keys.items():
            if k in have:
                continue
            ce = make_chemm_entry(s, target_mass)
            if ce is None or ce["ppm"] > FALLBACK_PPM:   # wrong formula -> discard
                continue
            ce["key"] = k; ce["chemm_exact"] = True
            entries.append(ce)

    for e in entries:
        e["mass_score"] = math.exp(-e["ppm"] / 5.0)
        e["anchor_score"] = anchor_similarity(e["smiles"], anchors)
        score = (W_MASS * e["mass_score"] + W_SPECTRAL * e["spectral_score"]
                 + W_CHEMMRL * e["anchor_score"] + W_PRIOR * e["prior_score"]
                 + SOURCE_BONUS.get(e["source"], 0.0))
        if e["chemm_exact"]:
            score += BONUS_CHEMM_R1
        if e["source"] in ("chemm_r1", "both") and e["ppm"] <= 5:
            score += BONUS_CHEMM_R1
        e["fusion_score"] = score

    ranked = sorted(entries, key=lambda x: x["fusion_score"], reverse=True)
    # guaranteed slots: top-3 CheMM-supported candidates + any very strong spectral match
    forced = [x for x in ranked if x["source"] in ("chemm_r1", "both") and x["ppm"] <= PRIMARY_PPM][:3]
    forced += [x for x in sorted(entries, key=lambda x: x["spectral_score"], reverse=True)[:2]
               if x["spectral_score"] >= STRONG_SPECTRAL and x["ppm"] <= FALLBACK_PPM]
    forced_ids = set()
    forced_u = []
    for x in forced:
        if id(x) not in forced_ids:
            forced_ids.add(id(x)); forced_u.append(x)
    forced_u.sort(key=lambda x: x["fusion_score"], reverse=True)
    ordered = forced_u + [x for x in ranked if id(x) not in forced_ids]

    selected, seen = [], set()
    for x in ordered:
        k = x.get("key") or ikey14(x["smiles"]) or x["smiles"]
        if k in seen:
            continue
        seen.add(k); x["key"] = k
        selected.append(x)
        if len(selected) >= FINAL_K:
            break
    return selected

def write_submission(final_records, path=SUBMISSION_PATH):
    sample_sub = pd.read_csv(SAMPLE_SUB)
    id_col = "molecule_id"
    target_col = "smiles" if "smiles" in sample_sub.columns else \
                 next(c for c in sample_sub.columns if c != id_col)
    rows = []
    for mid in sample_sub[id_col]:
        guesses, seen = [], set()
        for c in final_records.get(mid, []):
            k = c.get("key") or ikey14(c["smiles"]) or c["smiles"]
            if k in seen:
                continue
            seen.add(k); guesses.append(c["smiles"])
            if len(guesses) >= FINAL_K:
                break
        rows.append(";".join(guesses) if guesses else "CCO")
    sample_sub[target_col] = rows
    sample_sub.to_csv(path, index=False)
    return sample_sub, target_col

def run_fusion(chemm_preds, anchors_by_mid, desc="Fusion"):
    out = {}
    for mid, tm in tqdm(target_mass_by_mid.items(), desc=desc):
        out[mid] = fuse_candidates(float(tm), mass_pool_by_mid.get(mid, []),
                                   chemm_preds.get(mid, {}).get("smiles", []),
                                   anchors_by_mid.get(mid, []))
    return out

print("Fusion + submission helpers ready")

In [ ]:
t0 = time.time()
mass_pool_by_mid = {}
for mid, tm in tqdm(target_mass_by_mid.items(), desc="Retrieval"):
    mass_pool_by_mid[mid] = retrieve_candidates(float(tm), query_cache[mid]["all"])

best_spec = {mid: max((c["spectral_score"] for c in pool), default=0.0)
             for mid, pool in mass_pool_by_mid.items()}
vals = np.array(list(best_spec.values()))
print(f"Retrieval done in {(time.time()-t0)/60:.1f} min | elapsed {elapsed_min():.1f} min")
print("Best spectral score per molecule - quantiles (10/25/50/75/90%):",
      np.round(np.quantile(vals, [0.1, 0.25, 0.5, 0.75, 0.9]), 3))
print(f"Molecules with a strong library match (>= {STRONG_SPECTRAL}): "
      f"{(vals >= STRONG_SPECTRAL).sum()} / {len(vals)}  "
      f"-> high numbers mean test molecules overlap with train")

# Safety net: a complete retrieval-only submission exists from this point on.
chemm_predictions, anchor_by_mid = {}, {}
prelim = run_fusion(chemm_predictions, anchor_by_mid, desc="Fusion (retrieval only)")
_sub, _col = write_submission(prelim)
print(f"Preliminary submission written: {SUBMISSION_PATH} ({len(_sub)} rows)")

In [ ]:
# Choose which molecules get an LLM call, lowest library confidence first (so if time
# runs short, the molecules dropped are the ones retrieval already handles well),
# and pre-render their spectrum images in parallel BEFORE any model is loaded.
chemm_predictions = {}
if os.path.exists(CHEMM_CACHE_PATH):                      # resume support
    with open(CHEMM_CACHE_PATH) as f:
        for line in f:
            try:
                rec = json.loads(line); chemm_predictions[rec["mid"]] = {"raw": rec["raw"], "smiles": rec["smiles"]}
            except Exception:
                pass
    print(f"Resumed {len(chemm_predictions)} cached CheMM predictions")

CHEMM_AVAILABLE = bool(USE_CHEMM and torch.cuda.is_available())
todo, chemm_images = [], {}
if CHEMM_AVAILABLE:
    todo = [mid for mid in target_mass_by_mid
            if query_cache[mid]["visual"]
            and mid not in chemm_predictions
            and best_spec.get(mid, 0.0) < CHEMM_SKIP_IF_SPECTRAL_ABOVE]
    todo.sort(key=lambda m: best_spec.get(m, 0.0))
    if CHEMM_MOLECULES_CAP:
        todo = todo[:CHEMM_MOLECULES_CAP]
    print(f"Molecules queued for CheMM-R1: {len(todo)} of {len(target_mass_by_mid)} "
          f"(skipped {sum(1 for m in target_mass_by_mid if best_spec.get(m,0)>=CHEMM_SKIP_IF_SPECTRAL_ABOVE)} confident library matches)")

    t0 = time.time()
    jobs, owners = [], []
    for mid in todo:
        for x in query_cache[mid]["visual"]:
            jobs.append((x["mz"], x["int"], x["precursor_mz"], x["adduct"])); owners.append(mid)
    try:
        import multiprocessing as mp
        with mp.get_context("fork").Pool(min(4, os.cpu_count() or 1)) as pool_:
            pngs = pool_.map(render_ms2_png, jobs, chunksize=8)
    except Exception as e:
        print(f"[WARN] parallel render failed ({e!r}); rendering sequentially")
        pngs = [render_ms2_png(j) for j in jobs]
    for mid, b in zip(owners, pngs):
        chemm_images.setdefault(mid, {"png": [], "meta": query_cache[mid]["visual"]})["png"].append(b)
    todo = [m for m in todo if m in chemm_images]
    print(f"Rendered {len(pngs)} images for {len(todo)} molecules in {time.time()-t0:.1f}s")
else:
    print("CheMM-R1 disabled or no GPU - retrieval-only pipeline.")

In [ ]:
SYSTEM_PROMPT = (
    "You are a chemistry expert. Analyze the provided MS2 spectrum and identify "
    "the neutral molecule. Be concise: provide a short analysis (2-3 sentences) "
    "and then immediately output the final SMILES in <answer>SMILES</answer> tags. "
    "Do not write a full report."
)
USER_TEMPLATE = (
    "Identify the neutral molecule represented by these MS2 spectra.\n\n"
    "Precursor m/z values and adducts:\n{meta}\n\n"
    "Provide a brief analysis, then the final SMILES in <answer>SMILES</answer>.\n"
)
RESCUE_SUFFIX = "\n\nFinal answer: <answer>"

def _valid_smiles(s):
    s = s.strip().strip("\"'").strip()
    if not s:
        return None
    s = s.split()[0]
    mol = MolFromSmiles(s)
    return MolToSmiles(mol) if mol is not None else None

def parse_answer(text):
    """Last valid <answer>SMILES</answer>; tolerates a missing closing tag (truncated / rescued output)."""
    if not text:
        return None
    for s in reversed(re.findall(r"<answer>\s*([^<]*)", text, flags=re.I)):
        smi = _valid_smiles(s)
        if smi:
            return smi
    for s in reversed(re.findall(r"<smiles>\s*([^<]*)", text, flags=re.I)):
        smi = _valid_smiles(s)
        if smi:
            return smi
    return None

def build_prompt(proc, images, visual_meta):
    content = [{"type": "image", "image": im} for im in images]
    content.append({"type": "text", "text": USER_TEMPLATE.format(
        meta="\n".join(f"- {x['precursor_mz']:.4f} {x['adduct']}" for x in visual_meta))})
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": content}]
    return proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.inference_mode()
def _generate(worker, prompts, images_flat, max_new_tokens, greedy=False):
    """One batched generate() call (left-padded). Each sequence stops on its own at </answer>.
    Sampling parameters are set explicitly so the model's bundled generation_config
    (near-greedy top_k/top_p and repetition_penalty=1.05 for Qwen2.5-VL) cannot silently
    override them; a repetition penalty also hurts SMILES, which repeat tokens a lot."""
    proc, model = worker["proc"], worker["model"]
    inputs = proc(text=prompts, images=images_flat, return_tensors="pt", padding=True).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    kw = dict(max_new_tokens=max_new_tokens, stop_strings=["</answer>"], tokenizer=proc.tokenizer,
              pad_token_id=proc.tokenizer.pad_token_id, eos_token_id=proc.tokenizer.eos_token_id,
              repetition_penalty=1.0)
    if greedy or CHEMM_TEMPERATURE <= 0:
        kw.update(do_sample=False)
    else:
        kw.update(do_sample=True, temperature=CHEMM_TEMPERATURE, top_p=CHEMM_TOP_P, top_k=CHEMM_TOP_K)
    out = model.generate(**inputs, **kw)
    new = out[:, prompt_len:]
    steps = int(new.shape[1])
    texts = proc.batch_decode(new, skip_special_tokens=True)
    del out, inputs, new
    return texts, steps

def process_batch(worker, items, max_new_tokens):
    """items = [(mid, [PIL images], visual_meta), ...] -> (texts, steps)"""
    proc = worker["proc"]
    prompts = [build_prompt(proc, imgs, meta) for _, imgs, meta in items]
    flat = [im for _, imgs, _ in items for im in imgs]
    texts, steps = _generate(worker, prompts, flat, max_new_tokens)
    # Rescue: reasoning cut off by the token cap -> force the answer tag and let the model finish.
    need = [i for i, t in enumerate(texts) if parse_answer(t) is None]
    if need and CHEMM_RESCUE_TOKENS > 0:
        r_prompts = [prompts[i] + texts[i].rstrip() + RESCUE_SUFFIX for i in need]
        r_flat = [im for i in need for im in items[i][1]]
        r_texts, _ = _generate(worker, r_prompts, r_flat, CHEMM_RESCUE_TOKENS, greedy=True)
        for k, i in enumerate(need):
            texts[i] = texts[i].rstrip() + RESCUE_SUFFIX + r_texts[k]
    return texts, steps

def safe_process(worker, items, cap):
    """OOM-safe: on CUDA OOM split the batch in half and retry."""
    try:
        return process_batch(worker, items, cap)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        if len(items) == 1:
            return [""], 1
        h = len(items) // 2
        t1, s1 = safe_process(worker, items[:h], cap)
        t2, s2 = safe_process(worker, items[h:], cap)
        return t1 + t2, max(s1, s2)

print("CheMM-R1 generation helpers ready")

In [ ]:
workers = []
step_time_init = 0.08          # seconds per generated token step (refined from measurements)

def load_worker(dev, dtype):
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    proc = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    proc.tokenizer.padding_side = "left"                 # required for batched generation
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=dtype, device_map={"": dev},   # one full copy per GPU, never sharded
        trust_remote_code=True, attn_implementation="sdpa").eval()
    return {"model": model, "proc": proc, "dev": dev}

def load_all(dtype, n_gpu):
    from concurrent.futures import ThreadPoolExecutor
    ok = []
    with ThreadPoolExecutor(max_workers=n_gpu) as ex:        # load both GPUs in parallel
        futs = [ex.submit(load_worker, d, dtype) for d in range(n_gpu)]
        for d, f in enumerate(futs):
            try:
                ok.append(f.result())
            except Exception as e:
                print(f"[WARN] could not load worker on cuda:{d}: {e!r}")
    return ok

def free_workers(ws):
    for w in ws:
        w["model"] = None; w["proc"] = None
    ws.clear(); gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def warmup(worker):
    """Tiny real generation: verifies the dtype produces sane text and measures step latency."""
    mid = todo[0]
    imgs = [png_to_image(b) for b in chemm_images[mid]["png"]]
    prompt = build_prompt(worker["proc"], imgs, chemm_images[mid]["meta"])
    torch.cuda.synchronize(worker["dev"]); t = time.time()
    texts, _ = _generate(worker, [prompt], imgs, 48, greedy=True)
    torch.cuda.synchronize(worker["dev"]); dt = time.time() - t
    sane = sum(ch.isalpha() for ch in texts[0]) >= 15      # fp16 overflow gives '!!!!' / empty text
    return sane, dt / 48, texts[0][:120]

if CHEMM_AVAILABLE and todo:
    t0 = time.time()
    n_gpu = max(1, min(torch.cuda.device_count(), CHEMM_MAX_GPUS))
    # T4 (sm_75) has no native bf16: bf16 there is emulated and several times slower than fp16.
    major = torch.cuda.get_device_capability(0)[0]
    dtype_order = [torch.bfloat16, torch.float16] if major >= 8 else [torch.float16, torch.bfloat16]
    for dtype in dtype_order:
        print(f"Loading {n_gpu} CheMM-R1 copy/copies as {dtype} ...")
        workers = load_all(dtype, n_gpu)
        good, step_ts = [], []
        for w in workers:
            try:
                sane, st, sample = warmup(w)
                print(f"  cuda:{w['dev']} warmup sane={sane} step={st*1000:.0f}ms/token | {sample!r}")
                if sane:
                    good.append(w); step_ts.append(st)
            except Exception as e:
                print(f"  cuda:{w['dev']} warmup failed: {e!r}")
        if good:
            for w in workers:
                if w not in good:
                    w["model"] = None
            workers = good; gc.collect(); torch.cuda.empty_cache()
            step_time_init = max(0.02, max(step_ts))      # conservative: includes prefill
            break
        print(f"[WARN] {dtype} produced no usable output; trying the next dtype")
        free_workers(workers)
    if not workers:
        print("[WARN] CheMM-R1 unavailable - retrieval-only pipeline.")
        CHEMM_AVAILABLE = False
    print(f"{len(workers)} worker(s) ready in {(time.time()-t0)/60:.1f} min | elapsed {elapsed_min():.1f} min")

In [ ]:
# Time-aware, batched, two-GPU generation.
#  * one worker thread per GPU pulls batches of molecules from a shared queue
#  * before every batch the token cap is recomputed from the time that is left, the number of
#    batches still queued and the measured seconds/token -> the run always ends before
#    CHEMM_STOP_AT, and time saved by early-stopping sequences is automatically re-spent
#  * every finished batch is appended to a jsonl file (crash/timeout safe, resumable)
generation_count = 0
generation_times = []
batch_log = []

if CHEMM_AVAILABLE and workers and todo:
    state = {"step_t": step_time_init}
    lock = threading.Lock()
    work_q = queue.Queue()
    for i in range(0, len(todo), CHEMM_BATCH_SIZE):
        work_q.put(todo[i:i + CHEMM_BATCH_SIZE])
    n_workers = len(workers)
    total_batches = work_q.qsize()
    cache_f = open(CHEMM_CACHE_PATH, "a")
    print(f"{len(todo)} molecules -> {total_batches} batches over {n_workers} worker(s) | "
          f"budget {(CHEMM_STOP_AT - (time.time()-NOTEBOOK_START))/60:.0f} min | elapsed {elapsed_min():.1f} min")

    def worker_loop(worker):
        global generation_count
        with torch.cuda.device(worker["dev"]):
            while True:
                try:
                    mids = work_q.get_nowait()
                except queue.Empty:
                    return
                remaining = CHEMM_STOP_AT - (time.time() - NOTEBOOK_START)
                with lock:
                    step_t = state["step_t"]
                    per_worker = math.ceil((work_q.qsize() + 1) / n_workers)
                if remaining < CHEMM_MIN_TOKENS * step_t * 1.15:
                    return                                   # not enough time for another batch
                cap = int(remaining / (per_worker * step_t * 1.10))
                cap = max(CHEMM_MIN_TOKENS, min(CHEMM_MAX_TOKENS, cap))
                items = [(m, [png_to_image(b) for b in chemm_images[m]["png"]], chemm_images[m]["meta"])
                         for m in mids]
                t_b = time.time()
                try:
                    texts, steps = safe_process(worker, items, cap)
                except Exception as e:
                    print(f"[WARN] batch failed on cuda:{worker['dev']}: {e!r}")
                    texts, steps = [""] * len(mids), 1
                dt = time.time() - t_b
                with lock:
                    for m, t in zip(mids, texts):
                        smi = parse_answer(t)
                        chemm_predictions[m] = {"raw": [t], "smiles": [smi] if smi else []}
                        cache_f.write(json.dumps({"mid": m, "raw": [t], "smiles": [smi] if smi else []}) + "\n")
                    cache_f.flush()
                    generation_count += len(texts)
                    generation_times.extend([dt / len(mids)] * len(mids))
                    batch_log.append((cap, steps, dt))
                    state["step_t"] = 0.7 * state["step_t"] + 0.3 * (dt / max(steps, 1))

    threads = [threading.Thread(target=worker_loop, args=(w,), daemon=True) for w in workers]
    t_start = time.time(); last_print = 0
    for th in threads:
        th.start()
    while any(th.is_alive() for th in threads):
        time.sleep(5)
        if time.time() - last_print > 60:
            last_print = time.time()
            with lock:
                done = sum(1 for m in todo if m in chemm_predictions)
                parsed = sum(1 for m in todo if m in chemm_predictions and chemm_predictions[m]["smiles"])
                st = state["step_t"]
            print(f"  CheMM {done}/{len(todo)} done | parsed {parsed} | {st*1000:.0f} ms/token | "
                  f"elapsed {elapsed_min():.1f} min | stop-at {CHEMM_STOP_AT/60:.0f} min")
    for th in threads:
        th.join()
    cache_f.close()

    done = sum(1 for m in todo if m in chemm_predictions)
    parsed_total = sum(1 for v in chemm_predictions.values() if v["smiles"])
    print(f"\nCheMM finished: {done}/{len(todo)} queued molecules processed | "
          f"{parsed_total} with a parsed SMILES | {(time.time()-t_start)/60:.1f} min")
    if done < len(todo):
        print(f"  {len(todo)-done} molecules left to retrieval-only (time budget)")
    free_workers(workers)
else:
    print("CheMM generation skipped.")
print(f"Elapsed: {elapsed_min():.1f} min")

In [ ]:
# ChemMRL runs AFTER generation (GPU memory is free again) and only encodes what it needs:
# molecules that actually have CheMM-R1 anchors.
anchor_by_mid = {}
for mid, pred in chemm_predictions.items():
    anchors = []
    for s in pred["smiles"]:
        m = MolFromSmiles(s)
        if m is not None:
            anchors.append(MolToSmiles(m))
    anchor_by_mid[mid] = list(dict.fromkeys(anchors))[:CHEMMRL_MAX_ANCHORS]

need_smiles = set()
for mid, anchors in anchor_by_mid.items():
    if anchors:
        need_smiles.update(c["smiles"] for c in mass_pool_by_mid.get(mid, []))
        need_smiles.update(anchors)
print(f"Molecules with anchors: {sum(bool(a) for a in anchor_by_mid.values())} | "
      f"unique structures for ChemMRL: {len(need_smiles):,}")

CHEMMRL_ENC_STOP = HARD_DEADLINE_SECONDS - 8 * 60
if USE_CHEMMRL and need_smiles and (time.time() - NOTEBOOK_START) < CHEMMRL_ENC_STOP:
    try:
        from sentence_transformers import SentenceTransformer
        local_model = next((p for p in CHEMMRL_LOCAL_PATHS if os.path.exists(p)), None)
        model_source = local_model if local_model else CHEMMRL_MODEL_ID
        major = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
        dtype = torch.bfloat16 if major >= 8 else torch.float16
        chem_mrl = SentenceTransformer(model_source, trust_remote_code=True,
                                       similarity_fn_name="cosine", model_kwargs={"torch_dtype": dtype})
        chem_mrl.eval()
        CHEMMRL_ACTIVE, CHEMMRL_API = True, "sentence_transformers"
        print(f"ChemMRL active: {model_source}")

        smiles_list = sorted(need_smiles)
        t0 = time.time(); CHUNK = 5000
        embs, n_done = [], 0
        for start in range(0, len(smiles_list), CHUNK):
            if time.time() - NOTEBOOK_START > CHEMMRL_ENC_STOP:
                print("Runtime guard reached during ChemMRL encoding.")
                break
            chunk = smiles_list[start:start + CHUNK]
            emb = np.asarray(chem_mrl.encode(chunk, batch_size=CHEMMRL_BATCH_SIZE, show_progress_bar=False,
                                             convert_to_numpy=True, normalize_embeddings=False), dtype=np.float32)
            if CHEMMRL_EMB_DIM and emb.shape[1] > CHEMMRL_EMB_DIM:
                emb = emb[:, :CHEMMRL_EMB_DIM]                       # Matryoshka truncation ...
            emb /= np.maximum(np.linalg.norm(emb, axis=1, keepdims=True), 1e-9)   # ... then re-normalise
            if not np.all(np.isfinite(emb)):
                print(f"[WARN] non-finite embeddings in chunk {start}; skipped")
                continue
            for s, v in zip(chunk, emb):
                chem_mrl_embeddings[s] = v
            n_done += len(chunk)
        print(f"ChemMRL embedded {n_done:,}/{len(smiles_list):,} in {time.time()-t0:.1f}s")
        CHEMMRL_ACTIVE = bool(chem_mrl_embeddings)
    except Exception as e:
        print(f"[WARN] ChemMRL unavailable ({e!r}); falling back to Morgan similarity to CheMM anchors")
        chem_mrl_embeddings, CHEMMRL_ACTIVE = {}, False
    finally:
        chem_mrl = None; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    print("ChemMRL skipped (disabled, nothing to encode, or out of time) - Morgan similarity is used instead.")
print(f"CHEMMRL_ACTIVE={CHEMMRL_ACTIVE} | elapsed {elapsed_min():.1f} min")

In [ ]:
final_records = run_fusion(chemm_predictions, anchor_by_mid, desc="Final fusion")
print(f"Fusion complete for {len(final_records)} molecules")

sample_sub, target_col = write_submission(final_records)
print("Submission:", SUBMISSION_PATH, "| rows:", len(sample_sub))
print("Guesses per row:", sample_sub[target_col].str.split(";").map(len).describe()[["min", "50%", "max"]].to_dict())

invalid = total = 0
for row in sample_sub[target_col]:
    for smi in row.split(";"):
        total += 1
        if MolFromSmiles(smi) is None:
            invalid += 1
print(f"Invalid SMILES: {invalid}/{total}")
assert len(sample_sub) > 0 and invalid == 0, "Submission validation failed"

In [ ]:
elapsed = time.time() - NOTEBOOK_START
source_counts = defaultdict(int)
spectral_hits = anchor_hits = chemm_hits = 0
for mid, vals in final_records.items():
    for c in vals:
        source_counts[c["source"]] += 1
        spectral_hits += c.get("spectral_score", 0) > 0
        anchor_hits += c.get("anchor_score", 0) > 0
        chemm_hits += bool(c.get("chemm_exact", False))

debug = {
    "runtime_seconds": elapsed, "runtime_hours": round(elapsed / 3600, 3),
    "deadline_hours": HARD_DEADLINE_HOURS,
    "molecules": len(target_mass_by_mid),
    "chemm_queued": len(todo), "chemm_generations": generation_count,
    "chemm_parsed": sum(bool(v["smiles"]) for v in chemm_predictions.values()),
    "chemm_avg_time_per_molecule_s": float(np.mean(generation_times)) if generation_times else None,
    "chemm_batches": len(batch_log),
    "chemm_avg_token_cap": float(np.mean([b[0] for b in batch_log])) if batch_log else None,
    "chemmrl_active": CHEMMRL_ACTIVE, "chemmrl_embedded": len(chem_mrl_embeddings),
    "spectral_library_structures": len(lib_smiles), "spectral_library_spectra": len(spec_mz),
    "candidates_with_spectral_score": int(spectral_hits),
    "candidates_with_anchor_score": int(anchor_hits),
    "candidates_with_chemm_exact": int(chemm_hits),
    "source_counts": dict(source_counts),
}
with open(DEBUG_PATH, "w") as f:
    json.dump(debug, f, indent=2, default=str)
print(json.dumps(debug, indent=2))
assert os.path.exists(SUBMISSION_PATH)
print(f"\nDone. Wall time: {elapsed/60:.1f} min (deadline: {HARD_DEADLINE_HOURS*60:.0f} min)")